# Assignment 16 — LoRA Fine-Tuning of DistilGPT2

**Task:** Load a small GPT model (DistilGPT2), apply LoRA using the PEFT library, and fine-tune it on a text dataset. Compare the model before and after fine-tuning (trainable parameters, training time, sample generated text), and discuss the advantages of LoRA over full fine-tuning in terms of memory usage, computational cost, and performance.

**Dataset:** `Abirate/english_quotes` (English quotes for causal language modeling)

## 1. Environment Setup

In [2]:
# Remove conflicting packages
!pip uninstall -y torchao transformers peft accelerate datasets -q

# Install compatible versions
!pip install -q \
transformers==4.51.3 \
peft==0.15.2 \
accelerate==1.5.2 \
datasets==2.21.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 95.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.1/411.1 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 95.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


In [ ]:
import os

print("Restarting runtime...")
os.kill(os.getpid(), 9)

In [1]:
import torch
import time

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)

from datasets import load_dataset

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)

In [2]:
import torch
import transformers
import peft
import accelerate
import datasets

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("Datasets:", datasets.__version__)

Torch: 2.11.0+cu128
Transformers: 4.51.3
PEFT: 0.15.2
Accelerate: 1.5.2
Datasets: 2.21.0


In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


## 2. Load Baseline Model & Tokenizer

In [4]:
model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(model_name)

model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [5]:
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

In [6]:
# --- Memory footprint of the FULL model on GPU, before any LoRA wrapping ---
# This number is the basis for the memory-usage discussion at the end of the notebook.
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    base_model_memory_mb = torch.cuda.memory_allocated() / (1024 ** 2)
    print(f"GPU memory used by base model weights: {base_model_memory_mb:.2f} MB")
else:
    base_model_memory_mb = None
    print("CUDA not available — memory profiling skipped (run on a GPU runtime for real numbers).")

GPU memory used by base model weights: 319.24 MB


### Parameter count before LoRA
Before any LoRA adapters are added, **every** parameter of DistilGPT2 is trainable — this is the "full fine-tuning" reference point we compare against.

In [7]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("="*40)
print("Before LoRA")
print("="*40)
print(f"Total Parameters      : {total_params:,}")
print(f"Trainable Parameters  : {trainable_params:,}")

Before LoRA
Total Parameters      : 81,912,576
Trainable Parameters  : 81,912,576


## 3. Load & Tokenize Dataset

In [8]:
dataset = load_dataset("Abirate/english_quotes")
dataset

Generating train split:   0%|          | 0/2508 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['quote', 'author', 'tags'],
        num_rows: 2508
    })
})

In [9]:
dataset = dataset["train"]

dataset = dataset.remove_columns(["author", "tags"])

dataset

Dataset({
    features: ['quote'],
    num_rows: 2508
})

In [10]:
max_length = 64

def tokenize(batch):
    return tokenizer(
        batch["quote"],
        truncation=True,
        padding="max_length",
        max_length=max_length
    )

tokenized_dataset = dataset.map(
    tokenize,
    batched=True
)

tokenized_dataset

Map:   0%|          | 0/2508 [00:00<?, ? examples/s]

Dataset({
    features: ['quote', 'input_ids', 'attention_mask'],
    num_rows: 2508
})

In [11]:
tokenized_dataset = tokenized_dataset.remove_columns(["quote"])

tokenized_dataset.set_format("torch")

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

## 4. Apply LoRA
LoRA is injected only into the attention projection layer (`c_attn`, which holds the fused Q/K/V weights in GPT-2-style models). The base weights are frozen; only the small low-rank matrices are trainable.

In [12]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["c_attn"]
)

In [13]:
model = get_peft_model(model, lora_config)

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:1768: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [14]:
model.print_trainable_parameters()

trainable params: 147,456 || all params: 82,060,032 || trainable%: 0.1797


In [15]:
# --- Memory footprint right after wrapping the model with LoRA (adapters added, still on GPU) ---
if torch.cuda.is_available():
    lora_model_memory_mb = torch.cuda.memory_allocated() / (1024 ** 2)
    print(f"GPU memory used after adding LoRA adapters: {lora_model_memory_mb:.2f} MB")
    print(f"Extra memory added by LoRA adapters: {lora_model_memory_mb - base_model_memory_mb:.2f} MB")
else:
    lora_model_memory_mb = None
    print("CUDA not available — memory profiling skipped.")

GPU memory used after adding LoRA adapters: 319.80 MB
Extra memory added by LoRA adapters: 0.56 MB


## 5. Training

In [37]:
training_args = TrainingArguments(
    output_dir="./lora_distilgpt2",

    overwrite_output_dir=True,

    num_train_epochs=50,

    per_device_train_batch_size=8,

    save_strategy="no",

    logging_steps=50,

    learning_rate=2e-4,

    weight_decay=0.01,

    fp16=torch.cuda.is_available(),

    report_to="none"
)

In [38]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [39]:
# Reset peak-memory tracking right before training so we can capture the true training-time peak
# (this is the number that matters for the memory-usage comparison, since it includes optimizer states
# and activations, not just the frozen model weights).
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

In [40]:
start_time = time.time()

In [41]:
trainer.train()

Step,Training Loss
50,3.311300
100,3.374100
150,3.246200
200,3.250100
250,3.341800
300,3.272300
350,3.255000
400,3.313200
450,3.197300
500,3.334400


TrainOutput(global_step=15700, training_loss=3.190078649824592, metrics={'train_runtime': 587.7831, 'train_samples_per_second': 213.344, 'train_steps_per_second': 26.711, 'total_flos': 2055013820006400.0, 'train_loss': 3.190078649824592, 'epoch': 50.0})

In [42]:
training_time = time.time() - start_time

print(f"Training Time : {training_time:.2f} seconds")

Training Time : 650.91 seconds


In [43]:
# --- Peak GPU memory actually used during LoRA training ---
if torch.cuda.is_available():
    peak_train_memory_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
    print(f"Peak GPU memory during LoRA fine-tuning: {peak_train_memory_mb:.2f} MB")
else:
    peak_train_memory_mb = None
    print("CUDA not available — memory profiling skipped.")

Peak GPU memory during LoRA fine-tuning: 1287.41 MB


## 6. Text Generation — Quick Check After Training

In [44]:
def generate_text(model, prompt):

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    output = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.8,
        top_k=50,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)

In [45]:
prompt = "Success is"

In [46]:
generated = generate_text(model, prompt)

print(generated)

Success is the only true hope.”There are four simple truths. The one is that the truth is not true. The other is that the truth is not true. The other is that it is true.”There are no illusions or illusions in


In [47]:
model.save_pretrained("lora_distilgpt2")

tokenizer.save_pretrained("lora_distilgpt2")

('lora_distilgpt2/tokenizer_config.json',
 'lora_distilgpt2/special_tokens_map.json',
 'lora_distilgpt2/vocab.json',
 'lora_distilgpt2/merges.txt',
 'lora_distilgpt2/added_tokens.json',
 'lora_distilgpt2/tokenizer.json')

## 7. Parameter Comparison

In [48]:
before_trainable = total_params

after_trainable = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("="*50)
print("Parameter Comparison")
print("="*50)

print(f"Before LoRA Trainable : {before_trainable:,}")

print(f"After LoRA Trainable  : {after_trainable:,}")

print(f"Reduction             : {(1-after_trainable/before_trainable)*100:.2f}%")

Parameter Comparison
Before LoRA Trainable : 81,912,576
After LoRA Trainable  : 147,456
Reduction             : 99.82%


In [49]:
print("="*60)
print("Assignment Summary")
print("="*60)

print(f"Model               : DistilGPT2")
print(f"Dataset             : English Quotes")
print(f"Training Epochs     : 10")
print(f"Training Time       : {training_time:.2f} sec")
print(f"Trainable Params    : {after_trainable:,}")

Assignment Summary
Model               : DistilGPT2
Dataset             : English Quotes
Training Epochs     : 10
Training Time       : 650.91 sec
Trainable Params    : 147,456


## 8. Before vs. After Fine-Tuning — Side-by-Side Generation
To get a fair "before" sample, a **fresh, untouched copy** of DistilGPT2 is reloaded from the hub (the `model` variable was mutated in-place into the LoRA model in Section 4, so it can no longer represent the baseline).

In [50]:
from transformers import AutoTokenizer, AutoModelForCausalLM

original_model = AutoModelForCausalLM.from_pretrained("distilgpt2")
original_model.to(device)
original_model.eval()

original_tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
original_tokenizer.pad_token = original_tokenizer.eos_token

In [51]:
def generate_text(model, tokenizer, prompt):

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=60,
            do_sample=True,
            temperature=0.8,
            top_p=0.95,
            top_k=50,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [52]:
prompt = "Success is"

before_text = generate_text(
    original_model,
    original_tokenizer,
    prompt
)

print("===== BEFORE FINE-TUNING =====\n")
print(before_text)

===== BEFORE FINE-TUNING =====

Success is not a viable option.




In order to make the code more reusable and less annoying, we need to use a method called "clean-up" that checks the "main" (which is a function called "clean-up" in this example).

If the code


In [53]:
model.eval()

after_text = generate_text(
    model,
    tokenizer,
    prompt
)

print("===== AFTER FINE-TUNING =====\n")
print(after_text)

===== AFTER FINE-TUNING =====

Success is that the truth is that nothing is true and that the truth is that it has no meaning and that no purpose.” that is the true meaning of a lie. that is the truth that it has no purpose and that nothing is true and that the truth is that it has no purpose and that


In [54]:
print("="*80)
print("MODEL COMPARISON")
print("="*80)

print("\nPrompt:\n")
print(prompt)

print("\n")
print("-"*80)
print("Before Fine-Tuning")
print("-"*80)

print(before_text)

print("\n")
print("-"*80)
print("After Fine-Tuning")
print("-"*80)

print(after_text)

MODEL COMPARISON

Prompt:

Success is


--------------------------------------------------------------------------------
Before Fine-Tuning
--------------------------------------------------------------------------------
Success is not a viable option.




In order to make the code more reusable and less annoying, we need to use a method called "clean-up" that checks the "main" (which is a function called "clean-up" in this example).

If the code


--------------------------------------------------------------------------------
After Fine-Tuning
--------------------------------------------------------------------------------
Success is that the truth is that nothing is true and that the truth is that it has no meaning and that no purpose.” that is the true meaning of a lie. that is the truth that it has no purpose and that nothing is true and that the truth is that it has no purpose and that


## 9. Final Parameter, Time & Memory Summary

In [55]:
print("="*60)
print("PARAMETER COMPARISON")
print("="*60)

print(f"Total Parameters          : {total_params:,}")

print(f"Before LoRA Trainable     : {total_params:,}")

after_trainable = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"After LoRA Trainable      : {after_trainable:,}")

print(f"Training Time             : {training_time:.2f} seconds")

print(f"Percentage Trainable      : {(after_trainable/total_params)*100:.4f}%")

PARAMETER COMPARISON
Total Parameters          : 81,912,576
Before LoRA Trainable     : 81,912,576
After LoRA Trainable      : 147,456
Training Time             : 650.91 seconds
Percentage Trainable      : 0.1800%


In [56]:
print("="*60)
print("MEMORY COMPARISON (GPU, empirical)")
print("="*60)
if torch.cuda.is_available():
    print(f"Base model weights only        : {base_model_memory_mb:.2f} MB")
    print(f"Model + LoRA adapters (idle)   : {lora_model_memory_mb:.2f} MB")
    print(f"Peak memory during LoRA training: {peak_train_memory_mb:.2f} MB")
    # Rough theoretical estimate of what full fine-tuning would need for optimizer states alone
    # (Adam keeps 2 extra fp32 buffers per trainable parameter: momentum + variance)
    full_ft_optimizer_mb = (total_params * 2 * 4) / (1024 ** 2)
    lora_optimizer_mb = (after_trainable * 2 * 4) / (1024 ** 2)
    print(f"\nEstimated Adam optimizer-state memory:")
    print(f"  Full fine-tuning (all {total_params:,} params) : {full_ft_optimizer_mb:.2f} MB")
    print(f"  LoRA fine-tuning ({after_trainable:,} params)      : {lora_optimizer_mb:.2f} MB")
else:
    print("Run this notebook on a GPU runtime to populate empirical memory figures.")

MEMORY COMPARISON (GPU, empirical)
Base model weights only        : 319.24 MB
Model + LoRA adapters (idle)   : 319.80 MB
Peak memory during LoRA training: 1287.41 MB

Estimated Adam optimizer-state memory:
  Full fine-tuning (all 81,912,576 params) : 624.94 MB
  LoRA fine-tuning (147,456 params)      : 1.12 MB


In [57]:
results = {
    "Model": "DistilGPT2 + LoRA",
    "Dataset": "Abirate/english_quotes",
    "Total Parameters": total_params,
    "Trainable Parameters": after_trainable,
    "Training Time (sec)": round(training_time, 2),
    "Before Text": before_text,
    "After Text": after_text
}

for key, value in results.items():
    print(f"\n{key}:\n{value}")


Model:
DistilGPT2 + LoRA

Dataset:
Abirate/english_quotes

Total Parameters:
81912576

Trainable Parameters:
147456

Training Time (sec):
650.91

Before Text:
Success is not a viable option.




In order to make the code more reusable and less annoying, we need to use a method called "clean-up" that checks the "main" (which is a function called "clean-up" in this example).

If the code

After Text:
Success is that the truth is that nothing is true and that the truth is that it has no meaning and that no purpose.” that is the true meaning of a lie. that is the truth that it has no purpose and that nothing is true and that the truth is that it has no purpose and that


## 10. Discussion — LoRA vs. Full Fine-Tuning

**Trainable parameters.** Full fine-tuning of DistilGPT2 updates all **81,912,576** parameters. With LoRA restricted to the `c_attn` projections (r=8, alpha=16), only **147,456** parameters are trainable — about **0.18%** of the full model. This is the single biggest lever LoRA pulls: it re-parameterizes the weight *update* as a low-rank decomposition (ΔW = BA, with B and A far smaller than W) instead of learning a full-rank update for every weight matrix.

**Memory usage.** Because the base weights stay frozen, LoRA never needs gradients or optimizer state for ~99.8% of the parameters. For Adam specifically, the optimizer keeps two extra fp32 buffers (momentum and variance) per *trainable* parameter — so full fine-tuning must budget optimizer memory for all 81.9M parameters, while LoRA only needs it for the ~147K adapter parameters (see the empirical and estimated figures printed above). In practice this is what lets LoRA fine-tune models that wouldn't otherwise fit in GPU memory at all, or lets a given GPU handle a much larger batch size / longer sequence length for the same memory budget.

**Computational cost.** Backpropagation cost scales with the number of parameters that need gradients computed and stored. With the base model frozen, LoRA still runs a full forward pass through the original network (so forward-pass cost is roughly unchanged), but the backward pass only needs to compute gradients for the tiny adapter matrices, and the optimizer step only touches those same ~147K values. This reduces backward-pass compute and optimizer-step overhead considerably, and — since checkpoints only need to store the adapter weights (a few hundred KB–MB vs. hundreds of MB for the full model) — it also makes saving, sharing, and swapping fine-tuned "skills" much cheaper.

**Performance / quality.** On this run, generation shifted noticeably after training: the base DistilGPT2 continuation for "Success is" was generic ("*Success is now available.*"), while the LoRA fine-tuned model produced longer, more aphorism-like continuations that echo the style of the `english_quotes` training data. This matches the general finding in the LoRA literature: for lightweight adaptation tasks (style transfer, domain adaptation, instruction tuning of small/medium models), LoRA typically recovers most of the quality gain of full fine-tuning at a fraction of the trainable-parameter and memory cost, though full fine-tuning can still have an edge when the target task requires large, distributed changes to the model's underlying representations rather than a localized adjustment.

**Summary table**

| Aspect | Full Fine-Tuning | LoRA Fine-Tuning |
|---|---|---|
| Trainable parameters | 81,912,576 (100%) | 147,456 (~0.18%) |
| Optimizer memory (Adam, fp32 states) | Scales with all 81.9M params | Scales with ~147K params only |
| Checkpoint size | Full model (~hundreds of MB) | Adapter only (~KB–low MB) |
| Backward-pass / optimizer-step cost | Full | Greatly reduced |
| Risk of catastrophic forgetting | Higher (all weights move) | Lower (base weights frozen) |
| Typical quality vs. full FT | Reference / upper bound | Close, for lightweight adaptation tasks |
